# HR Data Studio

## 1. Install the small set of required libraries

In [ ]:
!pip install -q "transformers>=4.46,<5" "accelerate>=1,<2" gradio pandas


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Prepare the Colab runtime

In [ ]:
import json
import re
import uuid
from datetime import datetime
from pathlib import Path

import gradio as gr
import pandas as pd
import torch
from google.colab import userdata
from transformers import AutoModelForCausalLM, AutoTokenizer

OUTPUT_FOLDER = Path("/content/hr_outputs")
OUTPUT_FOLDER.mkdir(exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a T4 GPU from Runtime → Change runtime type.")

print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


## 3. Describe the available HR fields


In [ ]:
FIELD_RULES = {
    "employee_id": "Unique employee ID assigned by Python",
    "full_name": "Fictional first and last name",
    "age": "Integer from 22 to 64",
    "gender": "Male, Female, or Non-Binary",
    "department": "Engineering, Human Resources, Sales, Marketing, Finance, Operations, Legal, or Product",
    "annual_salary_usd": "Integer from 45000 to 185000",
    "years_of_experience": "Integer from 1 to 28 and not greater than age minus 18",
}

AVAILABLE_FIELDS = list(FIELD_RULES)
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

## 4. Load Llama into the Colab GPU

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype="auto",
    device_map="auto",
    low_cpu_mem_usage=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Loaded", MODEL_ID, "on", next(model.parameters()).device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.2-3B-Instruct on cuda:0


## 5. Build a strict JSON Schema for the selected fields


In [ ]:
def parse_json_response(text):
    text = re.sub(r"```(?:json)?", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "").strip()
    start = text.find("{")
    end = text.rfind("}")
    return json.loads(text[start:end + 1])["records"]


def generate_batch(selected_fields, batch_size, used_names):
    model_fields = [field for field in selected_fields if field != "employee_id"]

    if not model_fields:
        return [{} for _ in range(batch_size)]

    rules = "\n".join(f"- {field}: {FIELD_RULES[field]}" for field in model_fields)
    excluded_names = ", ".join(used_names[-30:]) if used_names else "None"

    prompt = f"""Create exactly {batch_size} fictional HR employee records.

Fields and rules:
{rules}

Previously used names that must not be repeated:
{excluded_names}

Return only valid JSON in this exact shape:
{{"records": [{{"field": "value"}}]}}

Every record must contain exactly these fields: {', '.join(model_fields)}.
Use JSON integers for age, salary, and experience.
Do not include employee_id, explanations, or Markdown.
"""

    messages = [
        {"role": "system", "content": "Return only valid JSON containing fictional HR data."},
        {"role": "user", "content": prompt},
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(next(model.parameters()).device)
    input_length = inputs["input_ids"].shape[1]
    token_limit = min(2400, max(500, batch_size * len(model_fields) * 25))

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=token_limit,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(output[0, input_length:], skip_special_tokens=True)
    return parse_json_response(answer)


## 6. Validate records and prevent duplicates


In [ ]:
def clean_record(record, selected_fields, seen_names, seen_records, employee_number):
    cleaned = {field: record[field] for field in selected_fields if field != "employee_id"}

    if "full_name" in cleaned:
        cleaned["full_name"] = " ".join(str(cleaned["full_name"]).split())
        name_key = cleaned["full_name"].casefold()
        if len(cleaned["full_name"].split()) < 2 or name_key in seen_names:
            return None

    for field in ("age", "annual_salary_usd", "years_of_experience"):
        if field in cleaned and not isinstance(cleaned[field], int):
            return None

    if "age" in cleaned and not 22 <= cleaned["age"] <= 64:
        return None
    if "annual_salary_usd" in cleaned and not 45000 <= cleaned["annual_salary_usd"] <= 185000:
        return None
    if "years_of_experience" in cleaned and not 1 <= cleaned["years_of_experience"] <= 28:
        return None
    if "age" in cleaned and "years_of_experience" in cleaned:
        if cleaned["years_of_experience"] > cleaned["age"] - 18:
            return None

    if "gender" in cleaned and cleaned["gender"] not in {"Male", "Female", "Non-Binary"}:
        return None
    if "department" in cleaned and cleaned["department"] not in {
        "Engineering", "Human Resources", "Sales", "Marketing",
        "Finance", "Operations", "Legal", "Product",
    }:
        return None

    fingerprint = json.dumps(cleaned, sort_keys=True, ensure_ascii=False)
    if fingerprint and fingerprint in seen_records:
        return None

    if "employee_id" in selected_fields:
        cleaned = {"employee_id": f"EMP-{employee_number:05d}", **cleaned}

    cleaned = {field: cleaned[field] for field in selected_fields}

    if "full_name" in cleaned:
        seen_names.add(cleaned["full_name"].casefold())
    if fingerprint:
        seen_records.add(fingerprint)

    return cleaned

## 7. Generate the full dataset and export it


In [ ]:
def generate_dataset(selected_fields, record_count, progress=gr.Progress()):
    if not selected_fields:
        raise gr.Error("Select at least one field.")

    record_count = int(record_count)
    if not 1 <= record_count <= 100:
        raise gr.Error("Record count must be between 1 and 100.")

    batch_size = 10
    records = []
    used_names = []
    seen_names = set()
    seen_records = set()
    employee_number = 1001
    maximum_requests = ((record_count + batch_size - 1) // batch_size) * 3

    for _ in range(maximum_requests):
        if len(records) == record_count:
            break

        needed = min(batch_size, record_count - len(records))
        raw_records = generate_batch(selected_fields, needed, used_names)

        for raw_record in raw_records:
            if len(records) == record_count:
                break
            cleaned = clean_record(
                raw_record,
                selected_fields,
                seen_names,
                seen_records,
                employee_number,
            )
            if cleaned is None:
                continue
            records.append(cleaned)
            employee_number += 1
            if cleaned.get("full_name"):
                used_names.append(cleaned["full_name"])

        progress(len(records) / record_count, desc=f"Generated {len(records)} of {record_count}")

    if len(records) != record_count:
        raise gr.Error(f"Llama produced {len(records)} valid records instead of {record_count}. Run Generate again.")

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
    csv_path = OUTPUT_FOLDER / f"hr_dataset_{run_id}.csv"
    json_path = OUTPUT_FOLDER / f"hr_dataset_{run_id}.json"

    dataframe = pd.DataFrame(records, columns=selected_fields)
    dataframe.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")

    return dataframe, str(csv_path), str(json_path)

## 8. Create the Gradio interface


In [ ]:
with gr.Blocks(title="HR Data Studio") as demo:
    gr.Markdown("# HR Data Studio")
    gr.Markdown(
        "Choose the HR fields you need and generate a ready-to-download dataset."
    )
    fields_input = gr.CheckboxGroup(
        choices=AVAILABLE_FIELDS,
        value=AVAILABLE_FIELDS,
        label="Fields",
    )
    count_input = gr.Number(
        value=10,
        precision=0,
        label="Number of records (1-100)",
    )
    generate_button = gr.Button("Generate")

    table_output = gr.Dataframe(interactive=False)
    csv_output = gr.File(label="CSV")
    json_output = gr.File(label="JSON")

    generate_button.click(
        generate_dataset,
        inputs=[fields_input, count_input],
        outputs=[table_output, csv_output, json_output],
    )

## 9. Launch Gradio


In [ ]:
demo.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8341d866463acf6aed.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
